<a href="https://colab.research.google.com/github/enzobanin/AnaliseDeDadosTemperatura/blob/main/TempArInst.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np

In [2]:
from google.colab import drive
drive.mount('/content/drive')

path = "/content/drive/MyDrive/Colab Notebooks/TemperaturaArInstantânea"

Mounted at /content/drive


In [4]:
df = pd.read_csv(path + '/A519.csv', sep=';')
df.head(5)

,data,hora,pressao,radiacao,temp_inst,pto_orvalho_inst,umid_inst,vento_vel,Unnamed: 8
0,2013-12-03,0,949.3,-3.54,22.3,21.0,92.0,2.3,NaN
1,2013-12-03,100,949.7,-3.54,22.1,20.6,91.0,2.1,NaN
2,2013-12-03,200,949.4,-3.54,22.1,20.0,88.0,1.3,NaN
3,2013-12-03,300,949.1,-3.54,21.9,20.2,90.0,2.3,NaN
4,2013-12-03,400,948.6,-3.54,21.4,19.8,91.0,1.5,NaN


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 83232 entries, 0 to 83231
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   data              83232 non-null  object 
 1   hora              83232 non-null  int64  
 2   pressao           77953 non-null  float64
 3   radiacao          77953 non-null  float64
 4   temp_inst         77953 non-null  float64
 5   pto_orvalho_inst  77948 non-null  float64
 6   umid_inst         77947 non-null  float64
 7   vento_vel         77953 non-null  float64
 8   Unnamed: 8        0 non-null      float64
dtypes: float64(7), int64(1), object(1)
memory usage: 5.7+ MB


In [6]:
df[df['temp_inst'].isnull()]['data'].value_counts().sort_index()
#soma a quantidade de valores nulos por data - para verificarmos se os valores nulos
#estão agrupados ou separados

,count
data,
2014-02-23,9
2014-02-24,13
2014-02-25,21
2014-02-26,15
2014-02-27,14
...,...
2021-11-02,11
2021-11-03,12
2021-11-04,12


In [8]:
df.duplicated().sum()
#verifica duplicidade

np.int64(0)

In [11]:
df = df.dropna(subset=['temp_inst', 'pressao', 'radiacao', 'pto_orvalho_inst', 'umid_inst', 'vento_vel'])
df = df.drop(columns=['Unnamed: 8'])

In [12]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 77945 entries, 0 to 83231
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   data              77945 non-null  object 
 1   hora              77945 non-null  int64  
 2   pressao           77945 non-null  float64
 3   radiacao          77945 non-null  float64
 4   temp_inst         77945 non-null  float64
 5   pto_orvalho_inst  77945 non-null  float64
 6   umid_inst         77945 non-null  float64
 7   vento_vel         77945 non-null  float64
dtypes: float64(6), int64(1), object(1)
memory usage: 5.4+ MB


In [ ]:
#mudar a data que veio do csv de string para dataTime
df['data'] = pd.to_datetime(df['data'], format='%Y-%m-%d')
#vai retornar o dia do ano para calculo posterior
dia_do_ano = df['data'].dt.dayofyear
#como 'hora' veio no formato HHMM, precisamos alterar
hora_do_dia = df['hora'] // 100

#a ideia vai ser espalhar as 24 horas do dia em um circulo para o algoritmo
#que vai estudar, sabera como funciona as horas,
#sera como um relogio analogico

#vamos fazer coordenadas com seno e cosseno para simular os horarios
df['hora_sin'] = np.sin(2 * np.pi * hora_do_dia / 24)
df['hora_cos'] = np.cos(2 * np.pi * hora_do_dia / 24)
#mesma logica porem vai representar o ano e nao as horas
df['dia_ano_sin'] = np.sin(2 * np.pi * dia_do_ano / 365.25)
df['dia_ano_cos'] = np.cos(2 * np.pi * dia_do_ano / 365.25)

df = df.drop(columns=['data', 'hora'])